# 01_collectAPI_stations : Stations piézométriques autour de Valence

### **Descriptif du Notebook VERSION VS CODE** :

**Le projet :**
- Connexion à l'API Hub'Eau sur le jeu de données "Piézométrie" (https://hubeau.eaufrance.fr/page/api-piezometrie). Ce notebook ne fait qu'extraire et enregistrer les données brutes, sans aucune transformation.
- Données recherchées :
    - A. **ICI** : la liste des stations piézométriques de la zone → endpoint `/stations`
    - B. Notebook `01_collectAPI_chroniques` : le niveau d'eau des stations → endpoint `/chroniques`
- Zone géographique : le bassin de Valence (26000) = bbox « carré de 100 km centré sur Valence »

**Les étapes du Notebook :**

1. Imports
2. Configuration : variables du projet (.env) + constantes métier
3. Calcul de la bbox
4. Fonction d'appel à l'API : définie dans `src/hubeau.py` (gestion des erreurs + pagination)
5. Récupération des stations piézométriques
6. Sauvegarde des données brutes au format JSON (`data/raw/`) + vérification par relecture

**Sorties :**
- `data/raw/stations_extrait_le_AAAA-MM-JJ.json` : réponse brute de l'API + contexte de l'extraction (date, URL, paramètres)

Le contrôle qualité, la mise en DataFrame et les exports (CSV, GeoJSON) sont faits dans `02_nettoyage_description_stations`.

#### ========== Prérequis avant d'utiliser le notebook ==========

**Première installation (une seule fois)**
- Cloner le dépôt, puis dans le terminal, à la racine du projet :
  `python3 -m venv .venv` → `source .venv/bin/activate` → `pip install -r requirements.txt`
- Créer le fichier de variables : `cp .env.example .env`

**À chaque session de travail**
- Ouvrir le **dossier racine** `Hub-eau` dans VS Code (pas seulement le notebook)
- Récupérer les dernières modifications : `git pull`
- Vérifier que le **kernel** sélectionné (en haut à droite) est `.venv`
- Exécuter la cellule de vérification ci-dessous

**Pour installer une nouvelle bibliothèque**
- Activer le venv dans le terminal : `source .venv/bin/activate` (l'invite doit commencer par `(.venv)`)
- `pip install <bibliothèque>`, puis `pip freeze > requirements.txt`, puis commit

**Avant chaque commit**
- *Restart Kernel* + *Run All* : le notebook doit tourner en entier sans erreur
- `git status` : `.env`, `.venv/` et les fichiers de `data/` ne doivent jamais apparaître

#### ========== Réutiliser le notebook ==========

- **Pour modifier les coordonnées GPS de la ville point central + dimension de la BBox**  
=> cellule "2. Configuration" > --- B. Constantes métier---  

- **Pour modifier les paramètres techniques de l'API**  
=> cellule "2. Configuration" > --- C. Paramètres techniques de l'API ---  

- **Pour modifier le format d'extraction**
=> cellule "5. Récupération" > paramètre `"format"` (l'API /stations renvoie du JSON ; l'enregistrement en cellule 6 conserve ce format d'origine)

In [1]:
# Vérification de l'environnement : le kernel doit être celui du .venv du projet
import sys

print("Python utilisé :", sys.executable)
assert ".venv" in sys.executable, "⚠️ Mauvais kernel : sélectionner le .venv en haut à droite"

Python utilisé : /home/data-08/Documents/00-PROJETS/Hub-eau/.venv/bin/python


## **Appel API - phase A /stations**

In [10]:
# 1. Imports des bibliothèques

# Rechargement automatique des modules de src/ quand ils sont modifiés
%load_ext autoreload
%autoreload 2

# Bibliothèque standard (livrée avec Python)
import json                         # sauvegarde des données brutes
import os                           # lecture des variables d'environnement
import sys                          # chemin de recherche des modules
from datetime import datetime       # horodatage des fichiers sauvegardés
from pathlib import Path            # chemins de fichiers (data/raw/...)

# Bibliothèques externes (installées dans le .venv, listées dans requirements.txt)
from dotenv import load_dotenv      # chargement du fichier .env

# Modules locaux (dossier src/ à la racine du projet)
sys.path.append(str(Path.cwd().parent))     # rend la racine du projet visible pour Python
from src.hubeau import appeler_api, bbox_autour

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [4]:
# 2. Configuration

# --- A. Variables d'environnement (fichier .env) ---
load_dotenv()                                   # charge le .env (cherché en remontant les dossiers)

BASE_URL = os.getenv("HUBEAU_BASE_URL")
if BASE_URL is None:
    raise ValueError("HUBEAU_BASE_URL introuvable : vérifier que le fichier .env existe à la racine du projet")

URL_STATIONS = f"{BASE_URL}/stations"           # liste des stations piézométriques

# --- B. Constantes métier ---
VALENCE_LAT = 44.9334                           # latitude du centre de la zone (WGS84)
VALENCE_LON = 4.8924                            # longitude du centre de la zone (WGS84)
DEMI_COTE_KM = 50                               # demi-côté du carré => carré de 100 km

# --- C. Paramètres techniques de l'API ---
PAGE_SIZE = 1000                                # nombre de résultats par page
TIMEOUT_S = 30                                  # délai max d'attente d'une réponse (secondes)
PAUSE_S = 0.2                                   # pause entre deux pages (respect de l'API)

# --- D. Chemins du projet ---
PROJECT_ROOT = Path.cwd().parent                # le notebook est dans notebooks/ => on remonte d'un niveau
assert (PROJECT_ROOT / "requirements.txt").exists(), f"Racine du projet incorrecte : {PROJECT_ROOT}"

DATA_RAW = PROJECT_ROOT / "data" / "raw"

# --- Récapitulatif ---
print("URL stations     :", URL_STATIONS)
print("Centre           :", VALENCE_LAT, VALENCE_LON, f"(demi-côté {DEMI_COTE_KM} km)")
print("Racine projet    :", PROJECT_ROOT)
print("Données brutes   :", DATA_RAW)

URL stations     : https://hubeau.eaufrance.fr/api/v1/niveaux_nappes/stations
Centre           : 44.9334 4.8924 (demi-côté 50 km)
Racine projet    : /home/data-08/Documents/00-PROJETS/Hub-eau
Données brutes   : /home/data-08/Documents/00-PROJETS/Hub-eau/data/raw


#### 3. Zone d'étude : bbox autour de Valence

Carré de **100 km de côté** (demi-côté `DEMI_COTE_KM` = 50 km) centré sur Valence,
calculé avec `bbox_autour()` (voir `src/hubeau.py`).

In [5]:
# 3. Calcul de la bbox (fonction : src/hubeau.py)

BBOX = bbox_autour(VALENCE_LAT, VALENCE_LON, DEMI_COTE_KM)
print("BBox :", BBOX)

# Contrôle visuel de la zone sur une carte
min_lon, min_lat, max_lon, max_lat = BBOX.split(",")
print(f"Vérifier : https://bboxfinder.com/#{min_lat},{min_lon},{max_lat},{max_lon}")

BBox : 4.2579,44.4842,5.5269,45.3826
Vérifier : https://bboxfinder.com/#44.4842,4.2579,45.3826,5.5269


#### 4. Fonction d'appel à l'API

La fonction `appeler_api()` est définie dans **`src/hubeau.py`** (importée à l'étape 1).

Elle gère :
- la **pagination** : suit le lien `next` renvoyé par l'API jusqu'à la dernière page ;
- les **erreurs** : accepte les codes 200 (complet) et 206 (partiel), lève une erreur sinon ;
- un **délai d'attente** (`timeout_s`) et une **pause** entre deux pages (`pause_s`) ;
- une **sécurité** `max_pages` pour éviter un nombre d'appels excessif.

⚠️ Limite Hub'Eau : `page × size` ne peut pas dépasser 20 000 résultats par requête.


In [6]:
# 5. Récupération des stations piézométriques

params_stations = {
    "bbox": BBOX,           # zone géographique calculée à l'étape 3
    "size": PAGE_SIZE,      # nombre de résultats par page (configuration)
    "format": "json",
}
# NB : la documentation liste d'autres filtres possibles pour /stations, comme code_departement ou date_recherche.

stations = appeler_api(URL_STATIONS, params_stations,
                       timeout_s=TIMEOUT_S, pause_s=PAUSE_S)

print(f"\n{len(stations)} stations récupérées")

Page 1 : 176 résultats (176/176)

176 stations récupérées


In [7]:
# 5 bis. Aperçu d'une station (structure brute de l'API)
stations[0]

{'code_bss': '07702X0081/1',
 'urn_bss': 'http://services.ades.eaufrance.fr/pointeau/07702X0081/1',
 'date_debut_mesure': None,
 'date_fin_mesure': None,
 'code_commune_insee': '26325',
 'nom_commune': "Saint-Rambert-d'Albon",
 'x': 4.83728378,
 'y': 45.289165654,
 'codes_bdlisa': None,
 'urns_bdlisa': [],
 'geometry': {'type': 'Point',
  'crs': {'type': 'name',
   'properties': {'name': 'urn:ogc:def:crs:OGC:1.3:CRS84'}},
  'coordinates': [4.83728377989336, 45.2891656535815]},
 'bss_id': 'BSS001WLSH',
 'altitude_station': '164.0',
 'nb_mesures_piezo': 0,
 'code_departement': '26',
 'nom_departement': 'Drôme',
 'libelle_pe': None,
 'profondeur_investigation': 27.25,
 'codes_masse_eau_edl': None,
 'noms_masse_eau_edl': None,
 'urns_masse_eau_edl': [],
 'date_maj': 'Wed Jun 25 20:01:28 CEST 2025'}

 #### 6. Sauvegarde des données brutes 

In [8]:
# 6. Sauvegarde des données brutes (data/raw/)

date_extraction = datetime.now()
fichier_brut = DATA_RAW / f"stations_extrait_le_{date_extraction:%Y-%m-%d}.json"

# Données de l'API telles quelles + contexte de l'extraction (traçabilité)
contenu_brut = {
    "date_extraction": date_extraction.isoformat(timespec="seconds"),
    "url": URL_STATIONS,
    "params": params_stations,
    "nb_resultats": len(stations),
    "data": stations,
}

with open(fichier_brut, "w", encoding="utf-8") as f:
    json.dump(contenu_brut, f, ensure_ascii=False, indent=2)

print(f"✅ {len(stations)} stations sauvegardées dans : {fichier_brut}")

✅ 176 stations sauvegardées dans : /home/data-08/Documents/00-PROJETS/Hub-eau/data/raw/stations_extrait_le_2026-11-16.json


In [9]:
# Vérification : relecture du fichier sauvegardé
with open(fichier_brut, encoding="utf-8") as f:
    relu = json.load(f)

print("Date d'extraction :", relu["date_extraction"])
print("Nombre de stations :", len(relu["data"]))
assert len(relu["data"]) == len(stations), "⚠️ Le fichier relu ne correspond pas aux données récupérées"

Date d'extraction : 2026-11-16T15:52:14
Nombre de stations : 176
